# 🔗 LoRA 병합 (Merge) — Colab에서 실행

Fine-tuned LoRA 어댑터를 Base 모델에 완전히 병합하여  
로컬 NPU/ARC GPU에서 사용할 단일 모델 파일을 생성합니다.

### 실행 순서
1. Cell 1 — 환경 설치
2. Cell 2 — Google Drive 마운트
3. Cell 3 — LoRA 병합 실행 (A100, ~5분)
4. Cell 4 — 압축 후 로컬 다운로드

> 📌 **다운로드 크기**: ~3~4GB (압축 전) / ~1.5GB (zip 압축 후)

## 📦 Cell 1: 환경 설치

In [ ]:
!pip install -q transformers peft sentencepiece timm torchvision einops accelerate safetensors
!git clone -q https://github.com/Meituan-AutoML/MobileVLM /content/MobileVLM
import sys
sys.path.insert(0, '/content/MobileVLM')
print('✅ 환경 설치 완료')

## 📁 Cell 2: Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ── 경로 설정 (필요시 수정) ────────────────────────────────────
ADAPTER_DIR = Path('/content/drive/MyDrive/fall_dataset/mobilevlm_qlora_adapter')
MERGED_DIR  = Path('/content/mobilevlm_merged')
MODEL_NAME  = 'mtgv/MobileVLM_V2-1.7B'

MERGED_DIR.mkdir(exist_ok=True)

# 어댑터 확인
print(f'어댑터 폴더 존재: {ADAPTER_DIR.exists()}')
if ADAPTER_DIR.exists():
    print('어댑터 파일 목록:')
    for f in ADAPTER_DIR.iterdir():
        print(f'  ✅ {f.name}')

## 🔗 Cell 3: LoRA 병합 실행

> ⏱️ A100 기준 약 5분 소요

In [ ]:
import torch
from peft import PeftModel
from mobilevlm.model.mobilevlm import load_pretrained_model
from mobilevlm.utils import disable_torch_init

print('Step 1/4: Base 모델 로드 중...')
disable_torch_init()
tokenizer, model_base, image_processor, _ = load_pretrained_model(
    MODEL_NAME,
    load_8bit=False,
    load_4bit=False,
    device='cuda',
)
print(f'  Base 모델 dtype: {next(model_base.parameters()).dtype}')
print(f'  GPU 메모리: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

print('\nStep 2/4: LoRA 어댑터 적용 중...')
peft_model = PeftModel.from_pretrained(model_base, str(ADAPTER_DIR))
print('  LoRA 어댑터 로드 완료')

print('\nStep 3/4: LoRA 병합 중 (merge_and_unload)...')
merged_model = peft_model.merge_and_unload()
print('  병합 완료!')

print('\nStep 4/4: 병합 모델 저장 중...')
merged_model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))
print(f'\n✅ 저장 완료: {MERGED_DIR}')
print('저장된 파일:')
for f in MERGED_DIR.iterdir():
    size_mb = f.stat().st_size / 1024**2
    print(f'  {f.name} ({size_mb:.1f} MB)')

## 💾 Cell 4: 압축 후 다운로드

> ⏱️ 압축에 2~3분, 다운로드는 인터넷 속도에 따라 다름  
> 📦 압축 파일 크기: 약 1.5~2GB

In [ ]:
import shutil
from google.colab import files

ZIP_PATH = '/content/mobilevlm_merged'

print('압축 중... (시간 소요)')
shutil.make_archive(ZIP_PATH, 'zip', str(MERGED_DIR))
zip_size = Path(ZIP_PATH + '.zip').stat().st_size / 1024**3
print(f'✅ 압축 완료: {zip_size:.2f} GB')

print('\n다운로드 시작...')
files.download(ZIP_PATH + '.zip')
print('✅ 다운로드 완료!')
print()
print('=' * 50)
print('📌 로컬 사용 방법:')
print('1. mobilevlm_merged.zip 다운로드 완료 확인')
print('2. C:\\mobilevlm_merged 폴더에 압축 해제')
print('3. inference_npu.ipynb 또는 inference_arc_gpu.ipynb 실행')
print('=' * 50)

## 📌 (선택) Drive에도 저장
다음에 재사용하려면 Drive에도 백업해두세요.

In [ ]:
import shutil

DRIVE_SAVE = '/content/drive/MyDrive/fall_dataset/mobilevlm_merged'
print(f'Drive에 저장 중: {DRIVE_SAVE}')
shutil.copytree(str(MERGED_DIR), DRIVE_SAVE, dirs_exist_ok=True)
print('✅ Drive 저장 완료!')
print('   다음 실행 시 이 경로에서 바로 로드 가능합니다.')